# Chapter 7 — Build It Yourself: Optimizers

See why **initialization** matters, then implement **SGD**, **Momentum**, and **Adam** from their
update rules and race them on a tiny task. "✍️ Your turn", "▶️ Run this", "▶️ Check your work". ⚙️

## Step 1 — Setup and the initialization effect  ▶️ Run this

Generates a two-moons dataset and a small network, then shows the **loss at init**: a good init
starts near −log(1/2) ≈ 0.69 (unsure); a bad one starts confidently wrong.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_moons(n=400, noise=0.1, seed=0):
    g = torch.Generator().manual_seed(seed)
    t = torch.rand(n // 2, generator=g) * math.pi
    top = torch.stack([torch.cos(t), torch.sin(t)], 1)
    bot = torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], 1)
    X = torch.cat([top, bot]) + noise * torch.randn(n, 2, generator=g)
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)]).long()
    return X, y

def fresh_model(seed=1337):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 32), nn.Tanh(),
                         nn.Linear(32, 32), nn.Tanh(),
                         nn.Linear(32, 2))

X, y = make_moons()
print(f"a good init should start near -log(1/2) = {math.log(2):.3f}\n")
losses_at_init = {}
for scale, label in [(1.0, "good (default)"), (30.0, "bad (output x30)")]:
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 2))
    with torch.no_grad():
        m[-1].weight *= scale
    losses_at_init[label] = F.cross_entropy(m(X), y).item()
    print(f"  {label:16} loss at init {losses_at_init[label]:.3f}")

In [ ]:
# ▶️ Check your work
try:
    assert losses_at_init["bad (output x30)"] > 3 * losses_at_init["good (default)"], "bad init should start far higher"
    print("✅ A bad init starts confidently wrong (much higher loss) — wasted early steps.")
except Exception as e:
    print("❌", e)

## Step 2 — SGD  ✍️ Your turn

Fill the update rule: step each parameter a little way **downhill** — opposite its gradient,
scaled by the learning rate.

<details><summary>Stuck? Show the answer</summary>

```python
p -= self.lr * g
```
</details>

In [ ]:
class SGD:
    def __init__(self, lr=0.1):
        self.lr = lr
    @torch.no_grad()
    def step(self, params, grads):
        for p, g in zip(params, grads):
            pass    # ✍️ step p downhill: opposite the gradient, scaled by the learning rate

In [ ]:
# ▶️ Check your work
try:
    p = torch.tensor([1.0, 2.0])
    SGD(lr=0.1).step([p], [torch.tensor([0.5, 0.5])])
    assert torch.allclose(p, torch.tensor([0.95, 1.95])), "after one step p should move by -lr*g"
    print("✅ SGD works — one step downhill.")
except Exception as e:
    print("❌", e)

## Step 3 — Momentum  ✍️ Your turn

Give the optimizer velocity: blend the old velocity with the new gradient (keep `beta` of the
old, add the new), then step by the velocity.

<details><summary>Stuck? Show the answer</summary>

```python
self.v[i] = self.beta * self.v[i] + g
p -= self.lr * self.v[i]
```
</details>

In [ ]:
class Momentum:
    def __init__(self, lr=0.1, beta=0.9):
        self.lr, self.beta, self.v = lr, beta, None
    @torch.no_grad()
    def step(self, params, grads):
        if self.v is None:
            self.v = [torch.zeros_like(p) for p in params]
        for i, (p, g) in enumerate(zip(params, grads)):
            # ✍️ blend the velocity (keep beta of the old, add the new gradient), then step p by lr * velocity
            pass    # replace (2 lines)

In [ ]:
# ▶️ Check your work
try:
    p = torch.tensor([0.0])
    mom = Momentum(lr=0.1, beta=0.9)
    mom.step([p], [torch.tensor([1.0])])      # v=1.0  -> p=-0.1
    mom.step([p], [torch.tensor([1.0])])      # v=1.9  -> p=-0.29
    assert torch.allclose(p, torch.tensor([-0.29]), atol=1e-5), "velocity should accumulate across steps"
    print("✅ Momentum works — the velocity builds up (moved more than two plain SGD steps would).")
except Exception as e:
    print("❌", e)

## Step 4 — Adam 🔑  ✍️ Your turn

The heart of the chapter. Fill the **two moments** (running average of the gradient, and of the
squared gradient) and the **Adam step** (mean ÷ √variance, scaled by `lr`). The bias correction
is given. See lesson §4. **Watch out:** unlike the momentum rule in Step 3 (`beta*v + g`), Adam
weights the *new* gradient by `(1 - beta)` — e.g. `b1 * m + (1 - b1) * g`.

<details><summary>Stuck? Show the answer</summary>

```python
self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
# (bias correction given)
p -= self.lr * mhat / (vhat.sqrt() + self.eps)
```
</details>

In [ ]:
class Adam:
    def __init__(self, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = self.v = None
        self.t = 0
    @torch.no_grad()
    def step(self, params, grads):
        if self.m is None:
            self.m = [torch.zeros_like(p) for p in params]
            self.v = [torch.zeros_like(p) for p in params]
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.m[i]    # ✍️ #1: running average of the gradient
            self.v[i] = self.v[i]    # ✍️ #2: running average of the SQUARED gradient
            mhat = self.m[i] / (1 - self.b1 ** self.t)      # bias correction (given)
            vhat = self.v[i] / (1 - self.b2 ** self.t)
            p -= 0    # ✍️ #3: the Adam step — corrected mean (mhat) ÷ √(corrected variance, vhat), scaled by lr

In [ ]:
# ▶️ Check your work — your Adam should make the SAME updates as torch.optim.Adam
try:
    torch.manual_seed(0)
    po = torch.randn(5); pt = po.clone()
    ours = Adam(lr=0.1); topt = torch.optim.Adam([pt], lr=0.1)
    for _ in range(10):
        ours.step([po], [2 * po])
        pt.grad = 2 * pt
        topt.step()
    assert torch.allclose(po, pt, atol=1e-5), "your Adam doesn't match torch.optim.Adam — check the moments & step"
    print("✅ Adam works — it matches torch.optim.Adam exactly. You built the modern optimizer.")
except Exception as e:
    print("❌", e)

## Step 5 — Race them  ▶️ Run this

Train the same fresh network on two-moons with each optimizer and compare the final loss.

In [ ]:
def train_with(opt, steps=300):
    model = fresh_model()
    params = list(model.parameters())
    for _ in range(steps):
        loss = F.cross_entropy(model(X), y)
        model.zero_grad()
        loss.backward()
        opt.step(params, [p.grad for p in params])
    return loss.item()

results = {}
for name, opt in [("SGD", SGD(lr=0.5)), ("Momentum", Momentum(lr=0.1)), ("Adam", Adam(lr=0.05))]:
    results[name] = train_with(opt)
    print(f"  {name:9} final loss {results[name]:.4f}")

In [ ]:
# ▶️ Check your work
try:
    assert results["Adam"] < results["SGD"], "Adam should reach a lower loss than SGD in the same steps"
    print(f"✅ Adam ({results['Adam']:.4f}) beat SGD ({results['SGD']:.4f}) — adaptive per-parameter steps win.")
except Exception as e:
    print("❌", e)

## 🎉 You built the optimizers.

SGD, Momentum, Adam — from their update rules — plus you saw why initialization matters. Adam's
per-parameter adaptive step is why it's the default for every modern model. Next: the
[exercises](../exercises/) (fix a broken init, sweep the LR, add warmup+cosine) and the
[mini-project](../project/), *The Optimizer Showdown*. 👋